In [ ]:
import base64
import requests
from ultralytics import YOLO
from dao.BaseDao import BaseDao
import os
import cv2
import numpy as np
import shutil


In [ ]:
import os
results_with_question = dict()  # 存储存在疑问的结果
results_with_error = dict()  # 存储存在错误的结果
all_book_result_in_dict = dict()  # 存储所有书籍识别结果
# 清理旧的运行目录
if os.path.exists('./runs'):
    shutil.rmtree('./runs')
# 执行预处理步骤
dict_coordinate_data, sorted_coordinate_dicts, removed_id_dicts = preProcess()

# 请求并保存识别结果
print('确认请求 ...')

for name, sorted_dict in sorted_coordinate_dicts.items():
    all_book_result_in_list_dict = dict()  # 存储单个文件内所有书籍识别结果
    for key, xywh in sorted_dict:
        if key in removed_id_dicts[name]:  # 若ID已被移除，则跳过
            continue
        path = seq_to_filepath(name, key, result_dir)   # 构建文件路径  
        # 整合hsv_get函数
        hsv_get(path)
        hsv_path = 'out/between_regions_hsv.jpg'  # 由hsv_get函数保存的图像路径
        if os.path.exists(path):
            all_book_result_in_list_dict[key] = send_post_request(
                image_to_base64(path)
            ).json()  # 发送请求并获取响应JSON  存储识别结果
            # print('finish ' + str(key))
    all_book_result_in_dict[name] = all_book_result_in_list_dict  # 将单个文件的识别结果加入总结果字典

In [ ]:
def is_subsequence(a, b):
    """判断a是否是b的子序列"""
    sub_iter = iter(a)
    return all(char in sub_iter for char in b if char in b)

In [ ]:
is_subsequence("开始写吧", "开始")

In [ ]:
import base64
from dao.BaseDao import BaseDao
import cv2

class BookDao(BaseDao):
    """
    职位数据管理数据库操作类
    DAO：database access object
    """
    # 获取book表中所有记录
    def getBooks(self):
        sql = 'select * from book'
        result = self.execute(sql=sql)
        resultSet = self.fetchall()
        return resultSet

def findLabelFromName(resultSet, character):
    """
    根据字符在结果集中查找标签匹配度超过75%的记录
    返回num_info；否则返回-1
    """
    for item in resultSet:
        count = 0
        length = len(character)
        for i in character:
            if i in item['label']:
                count += 1
            if count / length > 0.75:
                return item['num_info']
    return -1

def save_base64_image(data, file_path):
    """
    将Base64编码的图像数据保存到指定文件路径
    """
    try:
        # 提取并解码Base64数据
        base64_data = data
        binary_data = base64.b64decode(base64_data)
        # 将二进制数据写入文件
        with open(file_path, 'wb') as f:
            f.write(binary_data)
        print("图片保存成功")
    except Exception as e:
        print(f"图片保存失败: {e}")


def image_to_base64(image_path):
    """将图像文件转换为Base64编码字符串"""
    with open(image_path, "rb") as image_file:
        # 读取图片文件内容
        image_data = image_file.read()
        # 将图片内容编码为 base64 格式
        base64_encoded = base64.b64encode(image_data)
        # 将 bytes 类型转换为字符串类型
        base64_encoded_str = base64_encoded.decode('utf-8')
        return base64_encoded_str

def seq_to_filepath(filename, id, result_dir):
    """
    根据文件名、序号及结果目录生成完整文件路径
    如果id为0，则生成的文件路径为result_dir加上filename以及.jpg扩展名。
    否则，生成的文件路径为result_dir加上filename、(id+1)（表示序列编号）以及.jpg扩展名。
    """
    if id == 0:
        filepath = result_dir + str(filename) + '.jpg'
    else:
        filepath = result_dir + str(filename) + str(id + 1) + '.jpg'
    return filepath



def draw_bounding_box(image_path, errors_box, question_box, output_path):
    """在图像上绘制边界框并保存"""
    # 读取图像
    image = cv2.imread(image_path)
    # 绘制错误框（红色，厚度为20）
    for err in errors_box:
        x_center, y_center, width, height = err
        # 计算方框的左上角和右下角坐标
        x1 = int(x_center - width / 2)
        y1 = int(y_center - height / 2)
        x2 = int(x_center + width / 2)
        y2 = int(y_center + height / 2)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), thickness=20)

    # 绘制问题框（绿色，厚度为10）
    for question in question_box:
        x_center, y_center, width, height = question
        # 计算方框的左上角和右下角坐标
        x1 = int(x_center - width / 2)
        y1 = int(y_center - height / 2)
        x2 = int(x_center + width / 2)
        y2 = int(y_center + height / 2)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), thickness=10)

    cv2.imwrite(output_path, image)


In [ ]:
print(image_to_base64('shuji.jpg'))

In [121]:
import cv2
import numpy as np
import base64

from utils.ImageExecute import image_to_base64
from utils.OCR import send_post_request
# from utils.OCR import send_post_request
# from utils.ImageExecute import image_to_base64

def hsv_get(image):
    """
    再次分割书脊获得书标区域
    """

    # 将图像转换为HSV颜色空间
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # 定义红色的HSV范围
    lower_red = np.array([0, 100, 100])    # 红色的低阈值
    upper_red = np.array([10, 255, 255])   # 红色的高阈值

    # 创建一个mask，其中红色区域为白色，其他区域为黑色
    mask = cv2.inRange(hsv, lower_red, upper_red)

    # 寻找红线区域的轮廓
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 设置上下偏移量
    y_offset_top = -15
    y_offset_bottom = 10

    # 在原始图像上绘制红线区域的轮廓（仅作为示例）
    if contours:
        # 对轮廓按面积排序，取最大的两个轮廓
        contours = sorted(contours, key=cv2.contourArea, reverse=True)[:2]

        # 获取两条红线的 y 坐标
        y_coords = []
        for contour in contours:
            _, y, _, _ = cv2.boundingRect(contour)
            y_coords.append(y)

        # 确定上下两条红线的 y 坐标并应用偏移量
        y_coords.sort()
        y_top = max(y_coords[0] - y_offset_top, 0)
        y_bottom = min(y_coords[1] + y_offset_bottom, image.shape[0])

        # 提取两条红线之间的区域
        between_region = image[y_top:y_bottom, :]
        cv2.imwrite("between_region.jpg", between_region)
        return between_region
    else:
        print("未找到红线区域，请调整阈值或检查图像")
        return 0
    
def preprocess_image(image):
    """
    对图像进行预处理，包括对比度增强、二值化和去噪。
    """
    # 增强对比度
    # image = cv2.imread(image)
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    image = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    # 转为灰度图
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 二值化
    _, binary_image = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 去噪
    denoised_image = cv2.fastNlMeansDenoising(binary_image, None, 30, 7, 21)
    # # 边缘增强
    # kernel = np.ones((2, 2), np.uint8)
    # edges_enhanced = cv2.dilate(denoised_image, kernel, iterations=1)  # 应用膨胀操作增强边缘

    return denoised_image



In [120]:
cv2.imwrite('./hsv/1.jpg', preprocess_image('./hsv/2.jpg')) 

True

In [97]:
def getSingleCallNum(image_path):
    """
    识别并获取单本书的索书号（包括符号“=-:./”）
    """
    # 读取分割后的索书号区域图像
    image = cv2.imread(image_path)

    #hsv分割
    hsv_image = hsv_get(image)
    #图像处理
    pre_image = preprocess_image(hsv_image)
    #z转为base64
    base_64 = image2base64(pre_image)
    #OCR识别
    response = send_post_request(base_64).json()['data']['raw_out']
    # 创建列表用于保存有效的字符
    ascii_codes = []
    chars = []

    for _, element, conf in response:
        if conf > 0.8:
            
            # 拆分并检查每个字符是否有效
            for char in element:
                char = char.upper()#转为大写字母
                if char.isupper() or char.isdigit() or char in "=-:./":
                    chars.append(char)
    print("处理前：",chars)

    # 处理第一个字符
    if chars and not chars[0].isalpha():
        # 找到与第一个字符最相似的字母
        most_similar_char = find_most_similar_char(chars[0])
        if most_similar_char is not None:
            chars[0] = most_similar_char
            print("处理后：",chars)
    # 将有效字符转换为ASCII码并添加到列表中
    ascii_codes.extend(ord(c) for c in chars)
    return ascii_codes

def find_most_similar_char(char):
    """
    寻找与给定字符最相似的大写字母
    """
    # 假设的相似度阈值
    similarity_threshold = 0.5
    # 假设的字母相似度字典
    similarity_dict = {
        '0': ('O', 0.9),
        '1': ('I', 0.8),
        '2': ('Z', 0.7),
        # 其他字符及其相似度
    }

    # 查找相似度最高的字母
    max_similarity = 0
    most_similar_char = None
    for digit, (letter, similarity) in similarity_dict.items():
        if char == digit and similarity > max_similarity and similarity > similarity_threshold:
            max_similarity = similarity
            most_similar_char = letter

    return most_similar_char

In [122]:
getSingleCallNum("./test_pic/22.jpg")

处理前： ['I', '2', '6', '7', '5', '8', '6', '=', '4']


[73, 50, 54, 55, 53, 56, 54, 61, 52]

- 识别索书号并转为ASCII码完成（书标分割出来后在进行图像处理效果更好）
- 下面要将其整合进源代码，完成根据ASCII码比较索书号的功能
- 然后是识别书名，怎么能在书名旁有其他文字时确定书名